# 01 数据抽样与数据理解

本 Notebook 完成第一阶段工作：从 3.67GB 原始淘宝用户行为数据中生成 100 万行样本，并基于样本完成字段理解、数据质量检查、时间字段转换、基础清洗、核心指标计算和简单可视化。

注意：原始 CSV 文件较大，严禁直接一次性读取全量数据。本阶段只使用 `nrows=1000000` 读取前 100 万行样本。


## 1. 导入库

导入数据处理、数值计算和可视化所需的基础库。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 设置图表风格，便于后续快速查看趋势
sns.set_theme(style="whitegrid")
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False


## 2. 设置路径

统一管理原始数据、样本数据和清洗后样本数据路径，避免在后续代码中重复写路径。


In [ ]:
raw_path = r"E:\ecommerce-user-growth-analysis\data\raw\UserBehavior.csv"
sample_path = r"E:\ecommerce-user-growth-analysis\data\sample\UserBehavior_sample_100w.csv"
processed_sample_path = r"E:\ecommerce-user-growth-analysis\data\processed\UserBehavior_sample_cleaned.csv"

# 确保样本目录和处理后数据目录存在
os.makedirs(os.path.dirname(sample_path), exist_ok=True)
os.makedirs(os.path.dirname(processed_sample_path), exist_ok=True)


## 3. 定义列名

原始 CSV 文件没有表头，需要手动指定字段名称。


In [ ]:
col_names = ["user_id", "item_id", "category_id", "behavior_type", "timestamp"]


## 4. 预览前 10 行

先读取前 10 行确认文件结构是否符合预期。这里不会加载全量数据。


In [ ]:
preview = pd.read_csv(raw_path, names=col_names, nrows=10)
display(preview)


## 5. 生成 100 万行样本文件

使用 `nrows=1000000` 只读取前 100 万行，保存为样本文件。该样本用于初步探索和流程验证，不代表删除或替换原始数据。


In [ ]:
# 如果样本文件已经存在，可以避免重复生成；如需重跑，可先删除样本文件
if os.path.exists(sample_path):
    sample_df = pd.read_csv(sample_path)
    print(f"样本文件已存在，直接读取：{sample_path}")
else:
    sample_df = pd.read_csv(raw_path, names=col_names, nrows=1000000)
    sample_df.to_csv(sample_path, index=False, encoding="utf-8-sig")
    print(f"已生成 100 万行样本文件：{sample_path}")

sample_df.head()


## 6. 样本数据基础理解

检查样本规模、字段类型、缺失值、重复值和行为类型分布。


In [ ]:
# 查看前 5 行
sample_df.head()


In [ ]:
# 查看样本行列规模
sample_df.shape


In [ ]:
# 查看字段类型和非空数量
sample_df.info()


In [ ]:
# 检查各字段缺失值数量
sample_df.isnull().sum()


In [ ]:
# 检查重复行数量
sample_df.duplicated().sum()


In [ ]:
# 查看行为类型分布
sample_df["behavior_type"].value_counts()


## 7. 转换时间戳

将 Unix 秒级时间戳转换为可读时间，并新增日期、小时、星期字段。


In [ ]:
sample_df["datetime"] = pd.to_datetime(sample_df["timestamp"], unit="s", errors="coerce")

# 新增日期、小时和星期字段，便于后续活跃规律分析
sample_df["date"] = sample_df["datetime"].dt.date
sample_df["hour"] = sample_df["datetime"].dt.hour
sample_df["weekday"] = sample_df["datetime"].dt.dayofweek

sample_df.head()


## 8. 检查时间范围

查看样本数据覆盖的最早和最晚行为时间，判断数据周期是否合理。


In [ ]:
sample_df["datetime"].min()


In [ ]:
sample_df["datetime"].max()


## 9. 清洗数据

删除关键字段缺失值和重复记录，只保留合法行为类型。


In [ ]:
# 删除关键字段缺失的数据
sample_df = sample_df.dropna(subset=["user_id", "item_id", "category_id", "behavior_type", "timestamp", "datetime"])

# 删除完全重复的行为记录
sample_df = sample_df.drop_duplicates()

# 只保留合法行为类型
valid_behaviors = ["pv", "fav", "cart", "buy"]
sample_df = sample_df[sample_df["behavior_type"].isin(valid_behaviors)]

sample_df.shape


## 10. 保存清洗后的样本数据

将清洗后的样本保存到 `data/processed`，供后续 Notebook、SQL 和看板使用。


In [ ]:
sample_df.to_csv(processed_sample_path, index=False, encoding="utf-8-sig")
print(f"已保存清洗后样本数据：{processed_sample_path}")


## 11. 基础指标计算

计算平台规模、行为结构和用户购买转化率等第一批核心指标。


In [ ]:
total_behaviors = len(sample_df)
unique_users = sample_df["user_id"].nunique()
unique_items = sample_df["item_id"].nunique()
unique_categories = sample_df["category_id"].nunique()

behavior_counts = sample_df["behavior_type"].value_counts()
pv_count = behavior_counts.get("pv", 0)
fav_count = behavior_counts.get("fav", 0)
cart_count = behavior_counts.get("cart", 0)
buy_count = behavior_counts.get("buy", 0)

buyer_count = sample_df.loc[sample_df["behavior_type"] == "buy", "user_id"].nunique()
user_purchase_conversion_rate = buyer_count / unique_users if unique_users else np.nan

metrics = pd.DataFrame({
    "指标": [
        "总行为数", "独立用户数", "独立商品数", "独立类目数",
        "pv 浏览量", "fav 收藏数", "cart 加购数", "buy 购买数",
        "购买用户数", "用户购买转化率"
    ],
    "数值": [
        total_behaviors, unique_users, unique_items, unique_categories,
        pv_count, fav_count, cart_count, buy_count,
        buyer_count, user_purchase_conversion_rate
    ]
})

metrics


## 12. 简单可视化

绘制行为类型数量柱状图、每小时行为数量折线图和每日行为数量折线图，快速观察行为结构和活跃规律。


In [ ]:
# 各行为类型数量柱状图
plt.figure(figsize=(8, 5))
behavior_order = ["pv", "fav", "cart", "buy"]
sns.countplot(data=sample_df, x="behavior_type", order=behavior_order)
plt.title("各行为类型数量")
plt.xlabel("行为类型")
plt.ylabel("行为数量")
plt.show()


In [ ]:
# 每小时行为数量折线图
hourly_behavior = sample_df.groupby("hour").size().reset_index(name="behavior_count")

plt.figure(figsize=(10, 5))
sns.lineplot(data=hourly_behavior, x="hour", y="behavior_count", marker="o")
plt.title("每小时行为数量")
plt.xlabel("小时")
plt.ylabel("行为数量")
plt.xticks(range(0, 24))
plt.show()


In [ ]:
# 每日行为数量折线图
daily_behavior = sample_df.groupby("date").size().reset_index(name="behavior_count")

plt.figure(figsize=(12, 5))
sns.lineplot(data=daily_behavior, x="date", y="behavior_count", marker="o")
plt.title("每日行为数量")
plt.xlabel("日期")
plt.ylabel("行为数量")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
